# 00 — Pull Drive Data

Reads `../data_manifest.yaml` and fetches any missing files from Google Drive into `../data/raw/`.

Small CSV/JSON files are already committed; this notebook is for the **large** files (TREC runs, .pkl) that are gitignored.

## How it works
Uses `gdown` against each entry's `drive_id`. If a file already exists at the expected `local_path` with the expected `size_bytes`, it's skipped.

## Pre-reqs
```
pip install gdown pyyaml
```
If the file is not publicly shared, you may be prompted to authenticate in your browser the first time.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from _helpers import THESIS_FIGURES, DATA_RAW
import yaml

manifest = yaml.safe_load((THESIS_FIGURES / 'data_manifest.yaml').read_text(encoding='utf-8'))
entries = manifest['datasets']
print(f'{len(entries)} entries in manifest')

In [ ]:
# Identify what's missing
missing_drive, missing_local = [], []
for e in entries:
    target = THESIS_FIGURES / e['local_path']
    if target.exists():
        continue
    if e['source'] == 'drive':
        missing_drive.append(e)
    elif e['source'] == 'local':
        missing_local.append(e)

print(f'Missing from Drive: {len(missing_drive)}')
for e in missing_drive:
    print(f"  - {e['name']} -> {e['local_path']}")
print(f'\nMissing local copies (run cell below to copy): {len(missing_local)}')
for e in missing_local:
    print(f"  - {e['name']} from {e['local_source']}")

In [ ]:
# Pull missing Drive files via gdown
import subprocess
for e in missing_drive:
    target = THESIS_FIGURES / e['local_path']
    target.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {e['name']} ({e.get('size_bytes', '?')} bytes)...")
    cmd = ['gdown', '--id', e['drive_id'], '-O', str(target)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode == 0:
        print(f'  OK')
    else:
        print(f'  FAILED: {r.stderr.strip()}')
        print(f"  Manual fallback: open https://drive.google.com/file/d/{e['drive_id']}/view and download to {target}")

In [ ]:
# Copy missing local-source files
import shutil
repo_root = THESIS_FIGURES.parent
for e in missing_local:
    src = repo_root / e['local_source']
    dst = THESIS_FIGURES / e['local_path']
    if not src.exists():
        print(f"  MISSING SOURCE: {src}")
        continue
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f"  copied {e['name']}")

In [ ]:
# Verify sizes
for e in entries:
    target = THESIS_FIGURES / e['local_path']
    if not target.exists():
        print(f"  MISSING: {e['name']}")
        continue
    actual = target.stat().st_size
    expected = e.get('size_bytes')
    ok = expected in (None, '~') or abs(actual - expected) < 100
    flag = 'OK ' if ok else 'SIZE MISMATCH'
    print(f"  {flag}  {e['name']:40s}  {actual} bytes (expected {expected})")